# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook walks through loading, inspecting, and analyzing the FAIR² dataset using the `mlcroissant` library, referencing all dataset components by their `@id` as recommended by FAIR metadata principles.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant --quiet

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
# Access metadata as an object (not as a dictionary)
print(f"Dataset Name: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

The FAIR² dataset may include multiple record sets, each with its own fields. Below, we enumerate all record set `@id`s and for each, the field `@id`s and their descriptions.

In [ ]:
# List all record set @id's
print("Available Record Sets:")
record_set_ids = [rs['@id'] for rs in dataset.metadata.to_json().get('recordSet', [])]
for i, rs_id in enumerate(record_set_ids):
    print(f"  {i+1}: {rs_id}")
if not record_set_ids:
    print("  No top-level recordSet in root metadata; attempting to list from all recordSets by searching metadata JSON.")
    # Fallback: search for record sets in the schema
    json_md = dataset.metadata.to_json()
    # recordSets often defined in the 'hasPart' list
    rs_list = []
    if 'hasPart' in json_md:
        for part in json_md['hasPart']:
            if part.get('@type') == 'RecordSet' or part.get('@type') == 'cr:RecordSet':
                rs_list.append(part)
    if rs_list:
        record_set_ids = [rs['@id'] for rs in rs_list]
        print(f"  Found by hasPart: {record_set_ids}")
    else:
        print("  No record sets found by fallback.")# For demonstration, show fields of the *first* record set if present
if record_set_ids:
    first_rs_id = record_set_ids[0]
    print(f"\nFields for record set {first_rs_id}:")
    record_sets = dataset.metadata.to_json().get('recordSet', [])
    first_rs = None
    for rs in record_sets:
        if rs['@id'] == first_rs_id:
            first_rs = rs
            break
    if not first_rs:
        # Fallback, try hasPart
        for part in json_md.get('hasPart', []):
            if part.get('@id', None) == first_rs_id:
                first_rs = part
                break
    if first_rs and 'field' in first_rs:
        fields = first_rs['field']
        for f in fields:
            print(f"  - {f['@id']}: {f.get('name', '---')}, {f.get('description', '')}")
    else:
        print("  No fields found in record set.")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for further analysis.

Replace the record set and field `@id`s with those found in the previous cell if you wish to extract a different table.

In [ ]:
# Let's grab all record sets by their @id
metadata_json = dataset.metadata.to_json()
if 'recordSet' in metadata_json and metadata_json['recordSet']:
    record_sets = [rs['@id'] for rs in metadata_json['recordSet']]
else:
    # Fallback: try to extract from hasPart
    record_sets = []
    if 'hasPart' in metadata_json:
        for part in metadata_json['hasPart']:
            if part.get('@type') in ('RecordSet','cr:RecordSet'):
                record_sets.append(part['@id'])

if not record_sets:
    raise ValueError("No record sets found in metadata. Unable to proceed.")

dataframes = {}
for record_set_id in record_sets:
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded DataFrame for {record_set_id}: shape = {df.shape}")
    except Exception as e:
        print(f"Could not load records for {record_set_id}: {e}")

# Display fields (columns) of the first successfully loaded record set
for rs_id, df in dataframes.items():
    print(f"\nFields for record set {rs_id}:")
    print(df.columns.tolist())
    display_df_id = rs_id
    break

# Show first 5 records
dataframes[display_df_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps (filtering, normalization, grouping, etc.).

Below, select a numeric field and a grouping field using their `@id`s from the previous overview, then filter, normalize, and group the data as illustrative steps.

In [ ]:
# Identify a numeric field for demo (update with a valid field for your data)
df = dataframes[display_df_id]
numeric_field = None
possible_numeric_fields = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
if possible_numeric_fields:
    numeric_field = possible_numeric_fields[0]
    print(f"Using column '{numeric_field}' for numeric EDA.")
else:
    print("No numeric columns found. EDA will be illustrative.")

if numeric_field is not None:
    threshold = df[numeric_field].mean() if df[numeric_field].notnull().any() else 0
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records ({numeric_field} > {threshold:.2f}): {filtered_df.shape[0]} records.")

    # Normalize numeric field
    norm_col = f"{numeric_field}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(filtered_df[[numeric_field, norm_col]].head())

    # Try to group by a likely categorical column
    group_field = None
    for col in df.columns:
        if df[col].dtype == object and col != numeric_field and df[col].nunique() < 10:
            group_field = col
            break
    if group_field:
        print(f"Grouping by {group_field}:")
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
        print(grouped_df.head())
    else:
        print("No suitable group field found.")
else:
    print("Please select a proper numeric field for meaningful analysis.")

## 5. Visualization
Visualize data distributions or relationships using the selected fields.

Below is an example visualization for the chosen numeric field. Update the field `@id`s as needed for your use case.

In [ ]:
import matplotlib.pyplot as plt

if numeric_field is not None:
    plt.figure(figsize=(8,4))
    df[numeric_field].hist(bins=30)
    plt.title(f"Histogram of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Frequency")
    plt.show()

    # If a grouping field exists, show boxplot
    if group_field:
        plt.figure(figsize=(8,4))
        df.boxplot(column=numeric_field, by=group_field)
        plt.title(f"Boxplot of {numeric_field} by {group_field}")
        plt.suptitle("")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.show()

## 6. Conclusion
In this notebook, we demonstrated how to load and explore the FAIR² dataset (Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells) using the `mlcroissant` library.

- All dataset entities (record sets, fields) were referenced by their `@id`.
- We listed all available record sets and fields, loaded tabular data into DataFrames, and ran basic EDA including filtering and normalization.
- We visualized the distribution of a chosen numeric field.

This approach can be generalized to other FAIR datasets described via the Croissant standard using `mlcroissant`.